## Cell 1 — Install Ollama and LangChain Ollama integrati

In [1]:
!sudo apt-get update -qq && sudo apt-get install -y -qq pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -qU langchain-ollama

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## Cell 2 — Start Ollama and download the Qwen model

In [2]:
# 2. Start Ollama server in the background and pull Qwen 3.5 (4B or 2B)
import subprocess
import time

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)

!ollama pull qwen3.5:4b   # Or: !ollama pull qwen3.5:2b

## Cell 3 — Install and synchronize Deep Agents dependencies

In [3]:
# 3.
%pip install -q -U --upgrade-strategy eager \
    deepagents langchain langchain-core langgraph langchain-ollama

## Cell 4 — Configure the Ollama chat model

In [4]:
# 4. Swap ChatGroq / ChatGoogleGenerativeAI for ChatOllama
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="qwen3.5:4b",   # Or "qwen3.5:2b"
    temperature=0,
    num_ctx=16384,        # Important: prevents truncating Deep Agents prompts & file tools
)

## Cell 5 — Test Deep Agents with a hello message

In [5]:
# 5.
from deepagents import create_deep_agent
from langchain_ollama import ChatOllama
from langchain.messages import HumanMessage

# Use the Ollama model already downloaded in your notebook
model = ChatOllama(
    model="qwen3.5:4b",
    base_url="http://localhost:11434",
    temperature=0,
    num_ctx=16384,
)

# Create a simple Deep Agent
agent = create_deep_agent(
    model=model,
    system_prompt="You are a friendly assistant. Respond briefly.",
)

# Test the Deep Agent
result = agent.invoke({
    "messages": [
        HumanMessage(content="Say hello to me in one short sentence.")
    ]
})

print(result["messages"][-1].text)


Hello!


## Cell 6 — Verify the installed Ollama model

In [6]:
!ollama list

NAME          ID              SIZE      MODIFIED      
qwen3.5:4b    d8b0f5e9760c    3.3 GB    2 minutes ago    


## Cell 7 — Test Ollama directly through LangChain

In [7]:
print(model.invoke("Say hello").content)

Hello! 👋 How can I help you today?
